# 6. Kruskal-Wallis ANOVA Analysis

This notebook analyzes whether sentiment scores differ significantly across the three time windows (pre-cancel, cancel, post-apology) for YouTube comment sentiment data.

## Research Question

**RQ1**: *How do audience sentiment trajectories on YouTube change across pre-cancel, cancel, and post-apology windows for South Korean influencer apology videos?*

## Method

- **Normality Test**: Shapiro-Wilk test to determine data distribution
- **Main Test**: Kruskal-Wallis H test (non-parametric alternative to ANOVA)
- **Post-Hoc**: Dunn's test with Bonferroni correction for pairwise comparisons
- **Effect Size**: Kruskal-Wallis Eta-squared (η²)
- **Data**: Sentiment scores across PRE, CANCEL, POST windows
- **Significance Threshold**: α = 0.05

## Interpretation Guidelines

### Shapiro-Wilk Test
| p-value | Interpretation |
|---|---|
| ≥ 0.05 | Normally distributed |
| < 0.05 | Skewed (Non-normal) |

### Kruskal-Wallis H Test
| p-value | Interpretation |
|---|---|
| ≥ 0.05 | No significant difference between groups |
| < 0.05 | Significant difference exists between groups |

### Effect Size (Eta-squared)
| η² Value | Interpretation |
|---|---|
| 0.01 - 0.06 | Small effect |
| 0.06 - 0.14 | Medium effect |
| > 0.14 | Large effect |

### Dunn's Post-Hoc Test
- Compare adjusted p-values between each pair of windows
- Bonferroni correction adjusts for multiple comparisons

**Shapiro-Wilk Test**
This is to determine if the data is normal or skewed to be able to determine what kind of statistical testing will be utilized for further investigation.

In [ ]:
import pandas as pd
from scipy import stats

print("--- SHAPIRO-WILK TEST: SENTIMENT SCORES ---")

sentiment_df = pd.read_csv('sentiment_checkpoint_128000.csv')

sample_size = min(5000, len(sentiment_df))
sentiment_sample = sentiment_df.sample(n=sample_size, random_state=42)

windows = sentiment_sample['window'].dropna().unique()

for window in windows:
    scores = sentiment_sample[sentiment_sample['window'] == window]['sentiment_score']

    # Run the Shapiro-Wilk test
    statistic, p_value = stats.shapiro(scores)

    distribution = "Normally Distributed" if p_value >= 0.05 else "Skewed (Non-Normal)"

    print(f"Window: {window.upper()}")
    print(f"  Test Statistic: {statistic:.4f}")
    print(f"  p-value: {p_value:.4g} -> {distribution}\n")

**Kruskal-Wallis H & Dunn's Post-Hoc Test**
Since the data is skewed, a kruskal-wallis H and Dunn's Post-Hoc Test is utilized to determine if sentiments across different windows significantly differ from each other.

In [ ]:
!pip install scikit-posthocs

import pandas as pd
from scipy import stats
import scikit_posthocs as sp

print("--- KRUSKAL-WALLIS H TEST ---")

# 1. Load the raw comment dataset
# Make sure the file path matches where you have the CSV saved
sentiment_df = pd.read_csv('sentiment_checkpoint_128000.csv')

# 2. Drop any rows where 'window' or 'sentiment_score' might be missing
clean_df = sentiment_df.dropna(subset=['window', 'sentiment_score'])

# 3. Separate the sentiment scores into three distinct groups based on the window
pre_scores = clean_df[clean_df['window'] == 'pre']['sentiment_score']
cancel_scores = clean_df[clean_df['window'] == 'cancel']['sentiment_score']
post_scores = clean_df[clean_df['window'] == 'post']['sentiment_score']

# 4. Run the Kruskal-Wallis H Test
kw_stat, kw_p_value = stats.kruskal(pre_scores, cancel_scores, post_scores)

print(f"Kruskal-Wallis H-statistic: {kw_stat:.4f}")
print(f"p-value: {kw_p_value:.4g}\n")

# 5. Interpret the Kruskal-Wallis result
if kw_p_value < 0.05:
    print("Result: SIGNIFICANT difference found in sentiment across the windows (p < 0.05).")
    print("Proceeding to Dunn's Post-Hoc Test to see WHICH windows differ...\n")

    print("--- DUNN'S POST-HOC TEST (with Bonferroni correction) ---")

    dunn_results = sp.posthoc_dunn(clean_df, val_col='sentiment_score', group_col='window', p_adjust='bonferroni')

    # Print the resulting p-value matrix
    print(dunn_results)

def calculate_kruskal_eta_sq(data, group_col, value_col):
    groups = [group[value_col].values for name, group in data.groupby(group_col)]
    h_stat, p_val = stats.kruskal(*groups)

    # 2. Get necessary values for the formula
    n = len(data)             # Total number of observations
    k = len(groups)          # Number of groups

    # 3. Calculate Eta-squared
    eta_sq = (h_stat - k + 1) / (n - k)

    return h_stat, p_val, eta_sq

### Kruskal-Wallis Eta-Squared Effect Size

This is used to determine how much impact the shift between windows is present

In [ ]:
def calculate_kruskal_eta_sq(data, group_col, value_col):
    # 1. Perform the Kruskal-Wallis H-test
    groups = [group[value_col].values for name, group in data.groupby(group_col)]
    h_stat, p_val = stats.kruskal(*groups)

    n = len(data)
    k = len(groups)

    # 3. Calculate Eta-squared
    eta_sq = (h_stat - k + 1) / (n - k)

    return h_stat, p_val, eta_sq

h, p, effect_size = calculate_kruskal_eta_sq(clean_df, 'window', 'sentiment_score')

print(f"Eta-squared: {effect_size:.4f}")

Median is computed to see which of the windows differs the most

In [ ]:
median_sentiment_by_window = clean_df.groupby('window')['sentiment_score'].median()
display(median_sentiment_by_window)

## Summary

### Key Findings

1. **Normality Check**: Shapiro-Wilk test showed data is non-normal across all windows (p < 0.05)
2. **Main Test**: Kruskal-Wallis H test revealed significant differences in sentiment across PRE, CANCEL, and POST windows
3. **Post-Hoc**: Dunn's test with Bonferroni correction identifies which specific window pairs differ
4. **Effect Size**: Kruskal-Wallis Eta-squared (η²) measures the magnitude of the difference

### Interpretation

- The sentiment significantly differs across the three time windows
- Use Dunn's post-hoc results to identify which specific pairs (PRE vs CANCEL, PRE vs POST, CANCEL vs POST) have significant differences
- Effect size indicates practical significance of the difference

### Limitations

- Non-parametric tests are robust but may be less powerful than parametric alternatives
- Large sample size (128,000) may cause trivial differences to appear significant
- Effect size helps contextualize statistical significance